<a href="https://colab.research.google.com/github/SohaMoamen/AI_Agent_smart_city/blob/main/AI_Agent_meta_controller_smart_city.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =============================================================================
# merge_proposals.py
#
# Hierarchical Reinforcement Learning for Smart City
#
# Merge proposals generated by:
#   1. Traffic Agent
#   2. Pollution Agent
#   3. Energy Agent
#
# The script:
#   • Loads the three proposal files
#   • Synchronizes timestamps
#   • Removes unmatched snapshots
#   • Normalizes rewards and priorities
#   • Generates a unified Meta-Training Dataset
#
# Author : Dr. Soha Abdelmoamen
# =============================================================================
# ==========================================================
# Mount Google Drive
# ==========================================================

from google.colab import drive

drive.mount('/content/drive')
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler


warnings.filterwarnings("ignore")

# =============================================================================
# Google Drive Paths
# =============================================================================

BASE_PATH = "/content/drive/MyDrive/AI Agent/3 Agents"

TRAFFIC_FILE = os.path.join(
    BASE_PATH,
    "traffic_agent_results.csv"
)

POLLUTION_FILE = os.path.join(
    BASE_PATH,
    "pollution_agent_results.csv"
)

ENERGY_FILE = os.path.join(
    BASE_PATH,
    "energy_agent_results.csv"
)

OUTPUT_FILE = os.path.join(
    BASE_PATH,
    "meta_training_dataset.csv"
)

# =============================================================================
# Load Proposal Files
# =============================================================================

print("=" * 70)
print("Loading Agent Proposal Files...")
print("=" * 70)

traffic_df = pd.read_csv(TRAFFIC_FILE)
pollution_df = pd.read_csv(POLLUTION_FILE)
energy_df = pd.read_csv(ENERGY_FILE)

print(f"Traffic   : {traffic_df.shape}")
print(f"Pollution : {pollution_df.shape}")
print(f"Energy    : {energy_df.shape}")

print("=" * 70)
# =============================================================================
# Part 2 - Verify and Prepare Proposal Files
# =============================================================================

print("\n" + "=" * 70)
print("Verifying Proposal Files...")
print("=" * 70)

# -----------------------------------------------------------------------------
# Required Columns
# -----------------------------------------------------------------------------

REQUIRED_COLUMNS = [

    "snapshot",
    "timestamp",
    "recommended_action",
    "expected_reward",
    "confidence",
    "priority"

]

# -----------------------------------------------------------------------------
# Verify Columns
# -----------------------------------------------------------------------------

for name, df in {

    "Traffic": traffic_df,
    "Pollution": pollution_df,
    "Energy": energy_df

}.items():

    missing = [

        col for col in REQUIRED_COLUMNS

        if col not in df.columns

    ]

    if len(missing) > 0:

        raise ValueError(

            f"{name} file is missing columns: {missing}"

        )

print("All required columns exist.")


# -----------------------------------------------------------------------------
# Convert Timestamp
# -----------------------------------------------------------------------------

for df in [

    traffic_df,
    pollution_df,
    energy_df

]:

    df["timestamp"] = pd.to_datetime(df["timestamp"])


# -----------------------------------------------------------------------------
# Remove Duplicate Timestamps
# -----------------------------------------------------------------------------

traffic_before = len(traffic_df)
pollution_before = len(pollution_df)
energy_before = len(energy_df)

traffic_df = traffic_df.drop_duplicates(
    subset="timestamp"
)

pollution_df = pollution_df.drop_duplicates(
    subset="timestamp"
)

energy_df = energy_df.drop_duplicates(
    subset="timestamp"
)

print(f"Traffic   duplicates removed : {traffic_before - len(traffic_df)}")
print(f"Pollution duplicates removed : {pollution_before - len(pollution_df)}")
print(f"Energy    duplicates removed : {energy_before - len(energy_df)}")


# -----------------------------------------------------------------------------
# Sort by Timestamp
# -----------------------------------------------------------------------------

traffic_df = traffic_df.sort_values(
    "timestamp"
).reset_index(drop=True)

pollution_df = pollution_df.sort_values(
    "timestamp"
).reset_index(drop=True)

energy_df = energy_df.sort_values(
    "timestamp"
).reset_index(drop=True)


# -----------------------------------------------------------------------------
# Display Time Range
# -----------------------------------------------------------------------------

print("\nTime Range")

print(
    f"Traffic   : {traffic_df['timestamp'].min()}  -->  "
    f"{traffic_df['timestamp'].max()}"
)

print(
    f"Pollution : {pollution_df['timestamp'].min()}  -->  "
    f"{pollution_df['timestamp'].max()}"
)

print(
    f"Energy    : {energy_df['timestamp'].min()}  -->  "
    f"{energy_df['timestamp'].max()}"
)

print("=" * 70)
# =============================================================================
# Part 3 - Synchronize Proposal Files
# =============================================================================

print("\n" + "=" * 70)
print("Synchronizing Proposal Files...")
print("=" * 70)

# -----------------------------------------------------------------------------
# Keep Only Required Columns
# -----------------------------------------------------------------------------

traffic = traffic_df[
    [
        "timestamp",
        "recommended_action",
        "expected_reward",
        "confidence",
        "priority"
    ]
].copy()

pollution = pollution_df[
    [
        "timestamp",
        "recommended_action",
        "expected_reward",
        "confidence",
        "priority"
    ]
].copy()

energy = energy_df[
    [
        "timestamp",
        "recommended_action",
        "expected_reward",
        "confidence",
        "priority"
    ]
].copy()


# -----------------------------------------------------------------------------
# Rename Columns
# -----------------------------------------------------------------------------

traffic.columns = [

    "timestamp",

    "traffic_action",
    "traffic_reward",
    "traffic_confidence",
    "traffic_priority"

]

pollution.columns = [

    "timestamp",

    "pollution_action",
    "pollution_reward",
    "pollution_confidence",
    "pollution_priority"

]

energy.columns = [

    "timestamp",

    "energy_action",
    "energy_reward",
    "energy_confidence",
    "energy_priority"

]
traffic = traffic.dropna(subset=["timestamp"])
pollution = pollution.dropna(subset=["timestamp"])
energy = energy.dropna(subset=["timestamp"])

# -----------------------------------------------------------------------------
# Inner Merge
# -----------------------------------------------------------------------------

meta_df = traffic.merge(

    pollution,

    on="timestamp",

    how="inner"

)

meta_df = meta_df.merge(

    energy,

    on="timestamp",

    how="inner"

)


# -----------------------------------------------------------------------------
# Sort
# -----------------------------------------------------------------------------

meta_df = meta_df.sort_values(

    "timestamp"

).reset_index(drop=True)


# -----------------------------------------------------------------------------
# Rebuild Snapshot Index
# -----------------------------------------------------------------------------

meta_df.insert(

    0,

    "snapshot",

    np.arange(len(meta_df))

)


# -----------------------------------------------------------------------------
# Statistics
# -----------------------------------------------------------------------------

print(f"Traffic Rows     : {len(traffic_df):,}")
print(f"Pollution Rows   : {len(pollution_df):,}")
print(f"Energy Rows      : {len(energy_df):,}")

print("-" * 70)

print(f"Common Snapshots : {len(meta_df):,}")

print("-" * 70)

print(f"Traffic Removed  : {len(traffic_df)-len(meta_df):,}")
print(f"Pollution Removed: {len(pollution_df)-len(meta_df):,}")
print(f"Energy Removed   : {len(energy_df)-len(meta_df):,}")

print("=" * 70)
# =============================================================================
# Part 4 - Feature Engineering
# =============================================================================

print("\n" + "=" * 70)
print("Feature Engineering...")
print("=" * 70)

# -----------------------------------------------------------------------------
# Normalize Rewards
# -----------------------------------------------------------------------------

reward_columns = [

    "traffic_reward",
    "pollution_reward",
    "energy_reward"

]

for column in reward_columns:

    scaler = MinMaxScaler()

    meta_df[column] = scaler.fit_transform(
        meta_df[[column]]
    )

print("✓ Rewards normalized.")


# -----------------------------------------------------------------------------
# Normalize Priorities
# -----------------------------------------------------------------------------

priority_columns = [

    "traffic_priority",
    "pollution_priority",
    "energy_priority"

]

for column in priority_columns:

    scaler = MinMaxScaler()

    meta_df[column] = scaler.fit_transform(
        meta_df[[column]]
    )

print("✓ Priorities normalized.")


# -----------------------------------------------------------------------------
# Time Features
# -----------------------------------------------------------------------------

meta_df["hour"] = meta_df["timestamp"].dt.hour

meta_df["day_of_week"] = meta_df["timestamp"].dt.dayofweek

meta_df["is_weekend"] = (

    meta_df["day_of_week"] >= 5

).astype(int)

print("✓ Time features extracted.")


# -----------------------------------------------------------------------------
# Dataset Information
# -----------------------------------------------------------------------------

print("-" * 70)

print("Current Dataset Shape :", meta_df.shape)

print("-" * 70)

print(meta_df.head())

print("=" * 70)
# =============================================================================
# Part 5 - Validate and Save Meta Training Dataset
# =============================================================================

print("\n" + "=" * 70)
print("Validating Meta Training Dataset...")
print("=" * 70)

# -----------------------------------------------------------------------------
# Check Missing Values
# -----------------------------------------------------------------------------

missing_values = meta_df.isnull().sum().sum()

if missing_values == 0:
    print("✓ No missing values found.")
else:
    print(f"✗ Missing values detected: {missing_values}")
    raise ValueError("Dataset contains missing values.")


# -----------------------------------------------------------------------------
# Check Duplicate Timestamps
# -----------------------------------------------------------------------------

duplicates = meta_df["timestamp"].duplicated().sum()

if duplicates == 0:
    print("✓ No duplicate timestamps.")
else:
    print(f"✗ Duplicate timestamps found: {duplicates}")
    raise ValueError("Duplicate timestamps detected.")


# -----------------------------------------------------------------------------
# Check Confidence Range
# -----------------------------------------------------------------------------

confidence_columns = [

    "traffic_confidence",
    "pollution_confidence",
    "energy_confidence"

]

for col in confidence_columns:

    if ((meta_df[col] < 0) | (meta_df[col] > 1)).any():

        raise ValueError(f"{col} is outside [0,1].")

print("✓ Confidence values are valid.")


# -----------------------------------------------------------------------------
# Check Reward Range
# -----------------------------------------------------------------------------

reward_columns = [

    "traffic_reward",
    "pollution_reward",
    "energy_reward"

]

for col in reward_columns:

    mn = meta_df[col].min()
    mx = meta_df[col].max()

    print(f"{col:25s}: [{mn:.3f}, {mx:.3f}]")


# -----------------------------------------------------------------------------
# Check Priority Range
# -----------------------------------------------------------------------------

priority_columns = [

    "traffic_priority",
    "pollution_priority",
    "energy_priority"

]

for col in priority_columns:

    mn = meta_df[col].min()
    mx = meta_df[col].max()

    print(f"{col:25s}: [{mn:.3f}, {mx:.3f}]")


# -----------------------------------------------------------------------------
# Save Dataset
# -----------------------------------------------------------------------------

meta_df.to_csv(

    OUTPUT_FILE,

    index=False

)

print("\n" + "=" * 70)
print("Meta Training Dataset Saved Successfully")
print("=" * 70)

print(f"Output File : {OUTPUT_FILE}")
print(f"Total Snapshots : {len(meta_df):,}")
print(f"Total Features  : {meta_df.shape[1]}")

print("=" * 70)

Mounted at /content/drive
Loading Agent Proposal Files...
Traffic   : (4333, 15)
Pollution : (4392, 8)
Energy    : (4334, 16)

Verifying Proposal Files...
All required columns exist.
Traffic   duplicates removed : 0
Pollution duplicates removed : 0
Energy    duplicates removed : 0

Time Range
Traffic   : 2014-08-01 07:40:00  -->  2014-09-30 23:20:00
Pollution : 2014-08-01 00:00:00  -->  2014-09-30 23:40:00
Energy    : 2014-08-01 07:40:00  -->  2014-09-30 23:40:00

Synchronizing Proposal Files...
Traffic Rows     : 4,333
Pollution Rows   : 4,392
Energy Rows      : 4,334
----------------------------------------------------------------------
Common Snapshots : 4,333
----------------------------------------------------------------------
Traffic Removed  : 0
Pollution Removed: 59
Energy Removed   : 1

Feature Engineering...
✓ Rewards normalized.
✓ Priorities normalized.
✓ Time features extracted.
----------------------------------------------------------------------
Current Dataset Shape : 

In [10]:
# =============================================================================
# MetaController.py
#
# Hierarchical Reinforcement Learning for Smart City
#
# Meta Controller (DQN)
#
# Author : Dr. Soha Abdelmoamen
# =============================================================================

import os
import random
import warnings

import numpy as np
import pandas as pd

from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim


warnings.filterwarnings("ignore")


# =============================================================================
# Device Configuration
# =============================================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else

    "cpu"

)

print("=" * 70)
print("Meta Controller")
print("=" * 70)

print(f"Device : {DEVICE}")

print("=" * 70)


# =============================================================================
# Google Drive Paths
# =============================================================================

BASE_PATH = "/content/drive/MyDrive/AI Agent/3 Agents"

DATASET_FILE = os.path.join(

    BASE_PATH,

    "meta_training_dataset.csv"

)

MODEL_PATH = os.path.join(

    BASE_PATH,

    "meta_controller_model.pth"

)

CHECKPOINT_PATH = os.path.join(

    BASE_PATH,

    "meta_controller_checkpoint.pth"

)


# =============================================================================
# Meta Controller Parameters
# =============================================================================

STATE_SIZE = 14

ACTION_SIZE = 7
STATE_COLUMNS = [

    "traffic_action",
    "traffic_reward",
    "traffic_confidence",
    "traffic_priority",

    "pollution_action",
    "pollution_reward",
    "pollution_confidence",
    "pollution_priority",

    "energy_action",
    "energy_reward",
    "energy_confidence",
    "energy_priority",

    "hour",
    "is_weekend"

]


# =============================================================================
# DQN Hyperparameters
# =============================================================================

LEARNING_RATE = 1e-3

GAMMA = 0.99

BATCH_SIZE = 64

MEMORY_SIZE = 10000

TARGET_UPDATE = 100


# =============================================================================
# Exploration Parameters
# =============================================================================

EPSILON = 1.0

EPSILON_MIN = 0.05

EPSILON_DECAY = 0.995


# =============================================================================
# Training Parameters
# =============================================================================

NUM_EPISODES = 300

MAX_STEPS = None


# =============================================================================
# Random Seed
# =============================================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)


print("Configuration Loaded Successfully")

print("=" * 70)
# =============================================================================
# Part 2 - Deep Q Network
# =============================================================================

class DQN(nn.Module):

    """
    Deep Q-Network for Meta Controller.
    """

    def __init__(
        self,
        state_size,
        action_size
    ):

        super(DQN, self).__init__()

        self.network = nn.Sequential(

            # -------------------------------------------------------------
            # Input Layer
            # -------------------------------------------------------------
            nn.Linear(state_size, 128),
            nn.LayerNorm(128),
            nn.ReLU(),

            # -------------------------------------------------------------
            # Hidden Layer 1
            # -------------------------------------------------------------
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),

            # -------------------------------------------------------------
            # Hidden Layer 2
            # -------------------------------------------------------------
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.ReLU(),

            # -------------------------------------------------------------
            # Output Layer
            # -------------------------------------------------------------
            nn.Linear(32, action_size)

        )

    # -------------------------------------------------------------------------
    # Forward Pass
    # -------------------------------------------------------------------------

    def forward(self, x):

        return self.network(x)


# =============================================================================
# Network Test
# =============================================================================

print("\nBuilding Deep Q-Network...")

test_model = DQN(
    STATE_SIZE,
    ACTION_SIZE
).to(DEVICE)

print(test_model)

total_params = sum(
    p.numel() for p in test_model.parameters()
)

trainable_params = sum(
    p.numel() for p in test_model.parameters()
    if p.requires_grad
)

print("-" * 70)
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print("=" * 70)
# =============================================================================
# Part 3 - Experience Replay Memory
# =============================================================================

class ReplayMemory:

    """
    Experience Replay Buffer
    """

    def __init__(self, capacity):

        self.memory = deque(maxlen=capacity)

    # -------------------------------------------------------------------------
    # Store Experience
    # -------------------------------------------------------------------------

    def push(

        self,

        state,

        action,

        reward,

        next_state,

        done

    ):

        self.memory.append(

            (

                state,

                action,

                reward,

                next_state,

                done

            )

        )

    # -------------------------------------------------------------------------
    # Sample Mini-Batch
    # -------------------------------------------------------------------------

    def sample(

        self,

        batch_size

    ):

        batch = random.sample(

            self.memory,

            batch_size

        )

        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.FloatTensor(
            np.array(states)
        ).to(DEVICE)

        actions = torch.LongTensor(
            actions
        ).unsqueeze(1).to(DEVICE)

        rewards = torch.FloatTensor(
            rewards
        ).unsqueeze(1).to(DEVICE)

        next_states = torch.FloatTensor(
            np.array(next_states)
        ).to(DEVICE)

        dones = torch.FloatTensor(
            dones
        ).unsqueeze(1).to(DEVICE)

        return (

            states,

            actions,

            rewards,

            next_states,

            dones

        )

    # -------------------------------------------------------------------------
    # Current Buffer Size
    # -------------------------------------------------------------------------

    def __len__(self):

        return len(self.memory)

    # -------------------------------------------------------------------------
    # Check Enough Samples
    # -------------------------------------------------------------------------

    def ready(

        self,

        batch_size

    ):

        return len(self.memory) >= batch_size


# =============================================================================
# Replay Memory Test
# =============================================================================

print("\nInitializing Replay Memory...")

memory = ReplayMemory(MEMORY_SIZE)

print(f"Replay Buffer Capacity : {MEMORY_SIZE:,}")

print("=" * 70)
# =============================================================================
# Part 4 - Meta Controller
# =============================================================================

class MetaController:

    """
    Meta Controller using Deep Q-Network (DQN)
    """

    # -------------------------------------------------------------------------
    # Constructor
    # -------------------------------------------------------------------------

    def __init__(self):

        print("\nInitializing Meta Controller...")
        print("-" * 70)

        # -------------------------------------------------------------
        # State & Action Space
        # -------------------------------------------------------------

        self.state_size = STATE_SIZE
        self.action_size = ACTION_SIZE

        # -------------------------------------------------------------
        # Hyperparameters
        # -------------------------------------------------------------

        self.gamma = GAMMA
        self.batch_size = BATCH_SIZE

        self.epsilon = EPSILON
        self.epsilon_min = EPSILON_MIN
        self.epsilon_decay = EPSILON_DECAY

        # -------------------------------------------------------------
        # Networks
        # -------------------------------------------------------------

        self.policy_net = DQN(

            self.state_size,
            self.action_size

        ).to(DEVICE)

        self.target_net = DQN(

            self.state_size,
            self.action_size

        ).to(DEVICE)

        self.target_net.load_state_dict(

            self.policy_net.state_dict()

        )

        self.target_net.eval()

        # -------------------------------------------------------------
        # Optimizer
        # -------------------------------------------------------------

        self.optimizer = optim.Adam(

            self.policy_net.parameters(),

            lr=LEARNING_RATE

        )

        # -------------------------------------------------------------
        # Loss Function
        # -------------------------------------------------------------

        self.criterion = nn.MSELoss()

        # -------------------------------------------------------------
        # Replay Memory
        # -------------------------------------------------------------

        self.memory = ReplayMemory(

            MEMORY_SIZE

        )

        # -------------------------------------------------------------
        # Load Dataset
        # -------------------------------------------------------------

        self.meta_df = pd.read_csv(

            DATASET_FILE

        )

        self.states = self.meta_df[

            STATE_COLUMNS

        ].astype(np.float32).values

        print(f"Dataset Loaded : {len(self.meta_df):,} snapshots")

        # -------------------------------------------------------------
        # Training Statistics
        # -------------------------------------------------------------

        self.total_steps = 0

        self.episode_rewards = []

        self.loss_history = []

        print("-" * 70)
        print("Meta Controller Ready")
        print("=" * 70)
# =============================================================================
# Part 5 - Dataset Interface
# =============================================================================

    # -------------------------------------------------------------------------
    # Reset Episode
    # -------------------------------------------------------------------------

    def reset(self):

        """
        Reset the environment to the first state.
        """

        self.current_step = 0

        return self.states[self.current_step]

    # -------------------------------------------------------------------------
    # Get Current State
    # -------------------------------------------------------------------------

    def get_state(self):

        """
        Return current state.
        """

        return self.states[self.current_step]

    # -------------------------------------------------------------------------
    # Get Next State
    # -------------------------------------------------------------------------

    def get_next_state(self):

        """
        Return next state.
        """

        next_index = self.current_step + 1

        if next_index >= len(self.states):

            return None

        return self.states[next_index]

    # -------------------------------------------------------------------------
    # Move to Next Step
    # -------------------------------------------------------------------------

    def step_forward(self):

        """
        Move to next snapshot.
        """

        self.current_step += 1

    # -------------------------------------------------------------------------
    # Check Episode End
    # -------------------------------------------------------------------------

    def is_done(self):

        """
        Check whether the episode has finished.
        """

        return self.current_step >= len(self.states) - 1

    # -------------------------------------------------------------------------
    # Dataset Information
    # -------------------------------------------------------------------------

    def dataset_size(self):

        """
        Number of snapshots.
        """

        return len(self.states)

    # -------------------------------------------------------------------------
    # Get Timestamp
    # -------------------------------------------------------------------------

    def get_timestamp(self):

        """
        Return current timestamp.
        """

        return self.meta_df.iloc[self.current_step]["timestamp"]
# =============================================================================
# Part 6 - Action Selection
# =============================================================================

    # -------------------------------------------------------------------------
    # Select Action (Epsilon-Greedy)
    # -------------------------------------------------------------------------

    def select_action(self, state):

        """
        Select an action using epsilon-greedy policy.
        """

        # -------------------------------------------------------------
        # Exploration
        # -------------------------------------------------------------

        if random.random() < self.epsilon:

            action = random.randint(

                0,

                self.action_size - 1

            )

            return action

        # -------------------------------------------------------------
        # Exploitation
        # -------------------------------------------------------------

        state = torch.FloatTensor(

            state

        ).unsqueeze(0).to(DEVICE)

        self.policy_net.eval()

        with torch.no_grad():

            q_values = self.policy_net(state)

        self.policy_net.train()

        action = torch.argmax(

            q_values,

            dim=1

        ).item()

        return action


    # -------------------------------------------------------------------------
    # Update Epsilon
    # -------------------------------------------------------------------------

    def update_epsilon(self):

        """
        Decay exploration rate.
        """

        if self.epsilon > self.epsilon_min:

            self.epsilon *= self.epsilon_decay

            self.epsilon = max(

                self.epsilon,

                self.epsilon_min

            )


    # -------------------------------------------------------------------------
    # Predict Q-values
    # -------------------------------------------------------------------------

    def predict_q_values(self, state):

        """
        Return Q-values for analysis.
        """

        state = torch.FloatTensor(

            state

        ).unsqueeze(0).to(DEVICE)

        self.policy_net.eval()

        with torch.no_grad():

            q_values = self.policy_net(state)

        self.policy_net.train()

        return q_values.squeeze(0).cpu().numpy()


# =============================================================================
# Action Mapping
# =============================================================================

    ACTION_NAMES = {0: "Traffic",1: "Pollution",2: "Energy",3: "Traffic + Pollution",4: "Pollution + Energy",5: "Traffic + Energy",6: "All Agents"}
    # =============================================================================
    # Part 7 - Experience Replay Training
    # =============================================================================

    # -------------------------------------------------------------------------
    # Train Policy Network
    # -------------------------------------------------------------------------

    def replay(self):

        """
        Train the policy network using experience replay.
        """

        # -------------------------------------------------------------
        # Check Replay Memory
        # -------------------------------------------------------------

        if not self.memory.ready(self.batch_size):

            return None

        # -------------------------------------------------------------
        # Sample Mini-Batch
        # -------------------------------------------------------------

        states, actions, rewards, next_states, dones = \
            self.memory.sample(self.batch_size)

        # -------------------------------------------------------------
        # Current Q Values
        # -------------------------------------------------------------

        current_q_values = self.policy_net(states).gather(

            1,

            actions

        )

        # -------------------------------------------------------------
        # Target Q Values
        # -------------------------------------------------------------

        with torch.no_grad():

            next_q_values = self.target_net(

                next_states

            ).max(

                dim=1,

                keepdim=True

            )[0]

            target_q_values = rewards + (

                1 - dones

            ) * self.gamma * next_q_values

        # -------------------------------------------------------------
        # Compute Loss
        # -------------------------------------------------------------

        loss = self.criterion(

            current_q_values,

            target_q_values

        )

        # -------------------------------------------------------------
        # Backpropagation
        # -------------------------------------------------------------

        self.optimizer.zero_grad()

        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(

            self.policy_net.parameters(),

            max_norm=1.0

        )

        self.optimizer.step()

        # -------------------------------------------------------------
        # Store Statistics
        # -------------------------------------------------------------

        self.loss_history.append(

            loss.item()

        )

        self.total_steps += 1

        # -------------------------------------------------------------
        # Update Target Network
        # -------------------------------------------------------------

        if self.total_steps % TARGET_UPDATE == 0:

            self.update_target_network()

        # -------------------------------------------------------------
        # Decay Epsilon
        # -------------------------------------------------------------

        self.update_epsilon()

        return loss.item()


    # -------------------------------------------------------------------------
    # Update Target Network
    # -------------------------------------------------------------------------

    def update_target_network(self):

        """
        Copy policy network weights to target network.
        """

        self.target_net.load_state_dict(

            self.policy_net.state_dict()

        )
# =============================================================================
# Part 8 - Save / Load Utilities
# =============================================================================

    # -------------------------------------------------------------------------
    # Save Trained Model
    # -------------------------------------------------------------------------

    def save_model(self):

        """
        Save trained policy network.
        """

        torch.save(

            self.policy_net.state_dict(),

            MODEL_PATH

        )

        print(f"\nModel saved successfully.")
        print(f"Path : {MODEL_PATH}")


    # -------------------------------------------------------------------------
    # Load Trained Model
    # -------------------------------------------------------------------------

    def load_model(self):

        """
        Load trained policy network.
        """

        if not os.path.exists(MODEL_PATH):

            print("No trained model found.")

            return

        self.policy_net.load_state_dict(

            torch.load(

                MODEL_PATH,

                map_location=DEVICE

            )

        )

        self.target_net.load_state_dict(

            self.policy_net.state_dict()

        )

        print("Model loaded successfully.")


    # -------------------------------------------------------------------------
    # Save Checkpoint
    # -------------------------------------------------------------------------

    def save_checkpoint(self):

        """
        Save training checkpoint.
        """

        checkpoint = {

            "policy_network": self.policy_net.state_dict(),

            "target_network": self.target_net.state_dict(),

            "optimizer": self.optimizer.state_dict(),

            "epsilon": self.epsilon,

            "total_steps": self.total_steps,

            "loss_history": self.loss_history,

            "episode_rewards": self.episode_rewards

        }

        torch.save(

            checkpoint,

            CHECKPOINT_PATH

        )

        print(f"Checkpoint saved : {CHECKPOINT_PATH}")


    # -------------------------------------------------------------------------
    # Load Checkpoint
    # -------------------------------------------------------------------------

    def load_checkpoint(self):

        """
        Resume training from checkpoint.
        """

        if not os.path.exists(CHECKPOINT_PATH):

            print("Checkpoint not found.")

            return

        checkpoint = torch.load(

            CHECKPOINT_PATH,

            map_location=DEVICE

        )

        self.policy_net.load_state_dict(

            checkpoint["policy_network"]

        )

        self.target_net.load_state_dict(

            checkpoint["target_network"]

        )

        self.optimizer.load_state_dict(

            checkpoint["optimizer"]

        )

        self.epsilon = checkpoint["epsilon"]

        self.total_steps = checkpoint["total_steps"]

        self.loss_history = checkpoint["loss_history"]

        self.episode_rewards = checkpoint["episode_rewards"]

        print("Checkpoint restored successfully.")


    # -------------------------------------------------------------------------
    # Print Controller Summary
    # -------------------------------------------------------------------------

    def summary(self):

        """
        Print controller information.
        """

        print("\n" + "=" * 70)

        print("Meta Controller Summary")

        print("=" * 70)

        print(f"Dataset Size     : {len(self.states):,}")

        print(f"State Size       : {self.state_size}")

        print(f"Action Size      : {self.action_size}")

        print(f"Replay Memory    : {len(self.memory):,}")

        print(f"Epsilon          : {self.epsilon:.4f}")

        print(f"Learning Rate    : {LEARNING_RATE}")

        print(f"Gamma            : {self.gamma}")

        print(f"Training Steps   : {self.total_steps}")

        print("=" * 70)
# =============================================================================
# Test Meta Controller
# =============================================================================

if __name__ == "__main__":

    controller = MetaController()

    controller.summary()




Meta Controller
Device : cpu
Configuration Loaded Successfully

Building Deep Q-Network...
DQN(
  (network): Sequential(
    (0): Linear(in_features=14, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (8): ReLU()
    (9): Linear(in_features=32, out_features=7, bias=True)
  )
)
----------------------------------------------------------------------
Total Parameters     : 12,935
Trainable Parameters : 12,935

Initializing Replay Memory...
Replay Buffer Capacity : 10,000

Initializing Meta Controller...
----------------------------------------------------------------------
Dataset Loaded : 4,333 snapshots
------------------------------------------------------------------

In [11]:
# =============================================================================
# MetaEnvironment.py
#
# Hierarchical Reinforcement Learning Environment
#
# Author : Dr. Soha Abdelmoamen
# =============================================================================

import os
import warnings

import numpy as np
import pandas as pd


warnings.filterwarnings("ignore")


print("=" * 70)
print("Meta Environment")
print("=" * 70)
# =============================================================================
# Part 2 - Configuration
# =============================================================================

# -----------------------------------------------------------------------------
# Google Drive Paths
# -----------------------------------------------------------------------------

BASE_PATH = "/content/drive/MyDrive/AI Agent/3 Agents"

DATASET_FILE = os.path.join(

    BASE_PATH,

    "meta_training_dataset.csv"

)


# -----------------------------------------------------------------------------
# State Features
# -----------------------------------------------------------------------------

STATE_COLUMNS = [

    "traffic_action",
    "traffic_reward",
    "traffic_confidence",
    "traffic_priority",

    "pollution_action",
    "pollution_reward",
    "pollution_confidence",
    "pollution_priority",

    "energy_action",
    "energy_reward",
    "energy_confidence",
    "energy_priority",

    "hour",
    "is_weekend"

]


# -----------------------------------------------------------------------------
# Environment Parameters
# -----------------------------------------------------------------------------

STATE_SIZE = len(STATE_COLUMNS)

ACTION_SIZE = 7


# -----------------------------------------------------------------------------
# Action Mapping
# -----------------------------------------------------------------------------

ACTION_NAMES = {

    0: "Traffic",

    1: "Pollution",

    2: "Energy",

    3: "Traffic + Pollution",

    4: "Pollution + Energy",

    5: "Traffic + Energy",

    6: "All Agents"

}


# -----------------------------------------------------------------------------
# Reward Weights
# -----------------------------------------------------------------------------

TRAFFIC_WEIGHT = 0.40

POLLUTION_WEIGHT = 0.35

ENERGY_WEIGHT = 0.25


# -----------------------------------------------------------------------------
# Environment Information
# -----------------------------------------------------------------------------

print("Environment Configuration Loaded")

print(f"Dataset : {DATASET_FILE}")

print(f"State Size : {STATE_SIZE}")

print(f"Action Size : {ACTION_SIZE}")

print("=" * 70)
# =============================================================================
# Part 3 - Meta Environment Class
# =============================================================================

class MetaEnvironment:

    """
    Hierarchical Reinforcement Learning Environment
    """

    # -------------------------------------------------------------------------
    # Constructor
    # -------------------------------------------------------------------------

    def __init__(self):

        print("\nInitializing Meta Environment...")
        print("-" * 70)

        # ---------------------------------------------------------------------
        # Load Meta Dataset
        # ---------------------------------------------------------------------

        self.meta_df = pd.read_csv(DATASET_FILE)

        # ---------------------------------------------------------------------
        # Build State Matrix
        # ---------------------------------------------------------------------

        self.states = self.meta_df[STATE_COLUMNS].astype(
            np.float32
        ).values

        # ---------------------------------------------------------------------
        # Environment Information
        # ---------------------------------------------------------------------

        self.num_states = len(self.states)

        self.current_step = 0

        self.state_size = STATE_SIZE

        self.action_size = ACTION_SIZE

        # ---------------------------------------------------------------------
        # Episode Statistics
        # ---------------------------------------------------------------------

        self.total_reward = 0.0

        self.episode_steps = 0

        # ---------------------------------------------------------------------
        # Current State
        # ---------------------------------------------------------------------

        self.current_state = self.states[self.current_step]

        # ---------------------------------------------------------------------
        # Environment Ready
        # ---------------------------------------------------------------------

        print(f"Dataset Loaded : {self.num_states:,} snapshots")

        print(f"State Size     : {self.state_size}")

        print(f"Action Size    : {self.action_size}")

        print("-" * 70)

        print("Meta Environment Ready")

        print("=" * 70)
# =============================================================================
# Part 4 - Reset Environment
# =============================================================================

    # -------------------------------------------------------------------------
    # Reset Environment
    # -------------------------------------------------------------------------

    def reset(self):

        """
        Reset the environment to the initial state.

        Returns
        -------
        numpy.ndarray
            Initial state.
        """

        # ---------------------------------------------------------------------
        # Reset Environment Variables
        # ---------------------------------------------------------------------

        self.current_step = 0

        self.episode_steps = 0

        self.total_reward = 0.0

        # ---------------------------------------------------------------------
        # Set Initial State
        # ---------------------------------------------------------------------

        self.current_state = self.states[self.current_step]

        return self.current_state.copy()
# =============================================================================
# Part 5 - Environment Step
# =============================================================================

    # -------------------------------------------------------------------------
    # Execute One Environment Step
    # -------------------------------------------------------------------------

    def step(self, action):
        """
        Execute one action in the environment.

        Parameters
        ----------
        action : int
            Action selected by the Meta Controller.

        Returns
        -------
        next_state : numpy.ndarray
        reward : float
        done : bool
        info : dict
        """

        # ---------------------------------------------------------------------
        # Validate Action
        # ---------------------------------------------------------------------

        if action < 0 or action >= self.action_size:

            raise ValueError(f"Invalid action: {action}")

        # ---------------------------------------------------------------------
        # Current State
        # ---------------------------------------------------------------------

        current_state = self.current_state.copy()

        # ---------------------------------------------------------------------
        # Decision Fusion
        # (Will be implemented later)
        # ---------------------------------------------------------------------

        fused_decision = action

        # ---------------------------------------------------------------------
        # Reward Calculation
        # (Temporary Placeholder)
        # ---------------------------------------------------------------------

        reward = 0.0

        # ---------------------------------------------------------------------
        # Update Statistics
        # ---------------------------------------------------------------------

        self.total_reward += reward

        self.episode_steps += 1

        # ---------------------------------------------------------------------
        # Move to Next State
        # ---------------------------------------------------------------------

        self.current_step += 1

        # ---------------------------------------------------------------------
        # Check End of Episode
        # ---------------------------------------------------------------------

        done = self.current_step >= self.num_states - 1

        if done:

            next_state = current_state.copy()

        else:

            self.current_state = self.states[self.current_step]

            next_state = self.current_state.copy()

        # ---------------------------------------------------------------------
        # Additional Information
        # ---------------------------------------------------------------------

        info = {

            "step": self.current_step,

            "action": action,

            "decision": fused_decision,

            "total_reward": self.total_reward

        }

        return next_state, reward, done, info
# =============================================================================
# Part 6 - Environment Utilities
# =============================================================================

    # -------------------------------------------------------------------------
    # Get Current State
    # -------------------------------------------------------------------------

    def get_state(self):
        """
        Return the current state.
        """

        return self.current_state.copy()

    # -------------------------------------------------------------------------
    # Get Current Step
    # -------------------------------------------------------------------------

    def get_step(self):
        """
        Return the current step index.
        """

        return self.current_step

    # -------------------------------------------------------------------------
    # Get Current Timestamp
    # -------------------------------------------------------------------------

    def get_timestamp(self):
        """
        Return the timestamp of the current state.
        """

        return self.meta_df.iloc[self.current_step]["timestamp"]

    # -------------------------------------------------------------------------
    # Episode Finished
    # -------------------------------------------------------------------------

    def is_done(self):
        """
        Check whether the current episode has finished.
        """

        return self.current_step >= self.num_states - 1

    # -------------------------------------------------------------------------
    # Dataset Size
    # -------------------------------------------------------------------------

    def dataset_size(self):
        """
        Return the number of available snapshots.
        """

        return self.num_states

    # -------------------------------------------------------------------------
    # Environment Summary
    # -------------------------------------------------------------------------

    def summary(self):
        """
        Print environment information.
        """

        print("\n" + "=" * 70)
        print("Meta Environment Summary")
        print("=" * 70)

        print(f"Dataset Size   : {self.num_states:,}")
        print(f"State Size     : {self.state_size}")
        print(f"Action Size    : {self.action_size}")
        print(f"Current Step   : {self.current_step}")
        print(f"Episode Reward : {self.total_reward:.4f}")

        print("=" * 70)
# =============================================================================
# Part 7 - Environment Interfaces
# =============================================================================

    # -------------------------------------------------------------------------
    # Set Decision Fusion Engine
    # -------------------------------------------------------------------------

    def set_fusion_engine(self, fusion_engine):
        """
        Register the Decision Fusion Engine.
        """

        self.fusion_engine = fusion_engine

    # -------------------------------------------------------------------------
    # Set Reward Engine
    # -------------------------------------------------------------------------

    def set_reward_engine(self, reward_engine):
        """
        Register the Meta Reward Engine.
        """

        self.reward_engine = reward_engine

    # -------------------------------------------------------------------------
    # Check Environment Readiness
    # -------------------------------------------------------------------------

    def is_ready(self):
        """
        Check whether all required components are attached.
        """

        return (
            hasattr(self, "fusion_engine") and
            hasattr(self, "reward_engine")
        )
# =============================================================================
# Part 8 - Environment Testing
# =============================================================================

if __name__ == "__main__":

    print("\n" + "=" * 70)
    print("Testing Meta Environment")
    print("=" * 70)

    # -------------------------------------------------------------------------
    # Create Environment
    # -------------------------------------------------------------------------

    env = MetaEnvironment()

    env.summary()

    # -------------------------------------------------------------------------
    # Reset Environment
    # -------------------------------------------------------------------------

    state = env.reset()

    print("\nEnvironment Reset Successfully")

    print(f"State Shape : {state.shape}")

    print(f"Dataset Size : {env.dataset_size():,}")

    print(f"Timestamp : {env.get_timestamp()}")

    # -------------------------------------------------------------------------
    # Execute One Random Step
    # -------------------------------------------------------------------------

    random_action = np.random.randint(0, ACTION_SIZE)

    print(f"\nRandom Action : {random_action}")

    next_state, reward, done, info = env.step(random_action)

    print("\nStep Executed Successfully")

    print(f"Reward : {reward}")

    print(f"Done : {done}")

    print(f"Next State Shape : {next_state.shape}")

    print("\nInfo Dictionary")

    for key, value in info.items():

        print(f"{key} : {value}")

    print("\nEnvironment Test Completed Successfully")

    print("=" * 70)

Meta Environment
Environment Configuration Loaded
Dataset : /content/drive/MyDrive/AI Agent/3 Agents/meta_training_dataset.csv
State Size : 14
Action Size : 7

Testing Meta Environment

Initializing Meta Environment...
----------------------------------------------------------------------
Dataset Loaded : 4,333 snapshots
State Size     : 14
Action Size    : 7
----------------------------------------------------------------------
Meta Environment Ready

Meta Environment Summary
Dataset Size   : 4,333
State Size     : 14
Action Size    : 7
Current Step   : 0
Episode Reward : 0.0000

Environment Reset Successfully
State Shape : (14,)
Dataset Size : 4,333
Timestamp : 2014-08-01 07:40:00

Random Action : 6

Step Executed Successfully
Reward : 0.0
Done : False
Next State Shape : (14,)

Info Dictionary
step : 1
action : 6
decision : 6
total_reward : 0.0

Environment Test Completed Successfully


In [ ]:
# =============================================================================
# DecisionFusionEngine.py
#
# Hierarchical Decision Fusion Engine
#
# Author : Dr. Soha Abdelmoamen
# =============================================================================

import warnings

import numpy as np


warnings.filterwarnings("ignore")


print("=" * 70)
print("Decision Fusion Engine")
print("=" * 70)